In [65]:
import pandas as pd
from pybiomart import Server
from tqdm import tqdm
import pandas as pd
import json

In [ ]:
def remove_nan_from_set(s):
    """Remove null instances from set
    
    :param s: selected set
    :return: set with no null values
    """
    return {x for x in s if pd.notna(x)}

def read_subset_genes_csv(location):
    """Create a df of primary gene symbol- alias symbol pairs

    :param location: file location
    return: a df of gene records
    """
    
    subset_genes_xxxx_df = pd.read_csv(
        location, index_col=[0],dtype={"NCBI_ID": str,"HGNC_ID":str})
    subset_genes_xxxx_df['primary_gene_symbol'] = subset_genes_xxxx_df['gene_symbol'].str.upper()
    subset_genes_xxxx_df.drop(["gene_symbol"], axis=1, inplace=True)
    subset_genes_xxxx_df['alias_symbol'] = subset_genes_xxxx_df['alias_symbol'].str.upper()
    return subset_genes_xxxx_df

def make_col_ortholog_match(recording_df, source_df, animal= str):
    """Check for ortholog matches in the primary gene symbol- alias symbol pairs. 
    Adds a T/F column for each pair. T if the alias is an ortholog from the specified animal and F if not

    :param recording_df: df that contains the primary gene symbol- alias symbol pairs
    :param source_df: df that contains the orthologs and their associated human genes
    :param animal: the animal from with the orthologs are being checked
    return: the number of primary gene symbol- alias symbol pairs where the alias is an ortholog from the specified animal
    """
    df = recording_df.copy()
    df[f'{animal} Match'] = df.apply(lambda row: 
                            any((source_df['Gene name'] == row['primary_gene_symbol']) 
                                & 
                                (source_df[f'{animal} gene name'] == row['alias_symbol'])), axis=1)
    print(f"Added column: {animal} Match")
    return df

def convert_all_columns_to_uppercase(df):
    """Convert gene symbols to all-caps. Diffferent species have differing capitalization requirements and this will standardize.

    :param df: DataFrame containing gene symbols of unknown capitalizations
    :return: a DataFrame with all gene symbols all-caps
    """
    for column in df.columns:
        if df[column].dtype == 'object':  # Check if the column type is object
            df[column] = df[column].str.upper()

    return df

def match_alias_to_ortholog(og_recording_df, source_df):
    """Apply the make_col_ortholog_match function to all animal columns in the DataFrame.

    :param og_recording_df: DataFrame containing the primary gene symbol- alias symbol pairs
    :param source_df: DataFrame containing the orthologs and their associated human genes
    :return: a DataFrame with all match columns added
    """
    source_df = source_df.dropna(subset=['Gene name'])

    source_df = convert_all_columns_to_uppercase(source_df)
    
    source_df = source_df.groupby('Gene name', as_index=False).agg(combine_rows)

    recording_df = og_recording_df.copy()
    recording_df.columns = recording_df.columns.str.strip().str.replace(r'\s+', ' ', regex=True)
    source_df.columns = source_df.columns.str.strip().str.replace(r'\s+', ' ', regex=True)

    animal_columns = [col for col in source_df.columns if 'gene name' in col and col != 'Gene name']
    
    true_counts = {}

    for animal in animal_columns:
        # Extract the animal name
        animal_name = animal.replace(' gene name', '')
        recording_df = make_col_ortholog_match(recording_df, source_df, animal_name)

        true_count = recording_df[f'{animal_name} Match'].sum()
        true_counts[animal_name] = true_count
    return recording_df, true_counts

### Gene Data Prep
Load in subsetted gene data using previously defined functions from notebook 4. Data output from notebook 1 is required for this workflow.

In [6]:
# Combine data from all sources
databases = ['ensg','hgnc','ncbi']
subset = pd.DataFrame()
for database in databases:
    tdf = read_subset_genes_csv(f'../output/subset_genes_{database}_df.csv')
    subset = pd.concat([subset, tdf], axis=0)

# Group
subset = subset.rename(columns={'gene_symbol': 'primary_gene_symbol'}).groupby(['primary_gene_symbol','alias_symbol'], as_index=False).agg({
    "HGNC_ID": lambda x: set(x),
    "ENSG_ID": lambda x: set(x),
    "NCBI_ID": lambda x: set(x)
})

# Remove NaN
subset['NCBI_ID'] = subset['NCBI_ID'].apply(remove_nan_from_set)
subset['ENSG_ID'] = subset['ENSG_ID'].apply(remove_nan_from_set)
subset['HGNC_ID'] = subset['HGNC_ID'].apply(remove_nan_from_set)

# Save
subset.to_hdf("../output/subset_genes_df.h5", key='df', mode='w')
subset.to_csv("../output/subset_genes_df.csv", index=True)

/var/folders/5t/sfw5tjx56m10xb861_pd3wfm0000gq/T/ipykernel_12097/1766384476.py:21: PerformanceWarning: 
your performance may suffer as PyTables will pickle object types that it cannot
map directly to c-types [inferred_type->mixed,key->block0_values] [items->Index(['primary_gene_symbol', 'alias_symbol', 'HGNC_ID', 'ENSG_ID', 'NCBI_ID'], dtype='object')]

  subset.to_hdf("../output/subset_genes_df.h5", key='df', mode='w')


### Download Ortholog Datasets
Programmatically download all ortholog gene name datasets from biomart using the pybiomart package. Run time takes ~15 minutes for the 200 orthologs available.

In [48]:
server = Server(host='http://www.ensembl.org')
dataset = server.marts['ENSEMBL_MART_ENSEMBL'].datasets['hsapiens_gene_ensembl']

homologs = []
for key in dataset.attributes.keys():
    if '_homolog_associated_gene_name' in key:
        homologs.append(key)
print(f'{len(homologs)} found!')

for homolog in tqdm(homologs):
    attributes = ['ensembl_gene_id', 'external_gene_name', homolog]
    df = dataset.query(attributes=attributes)
    df.to_csv(f'../input/orthologs/{homolog}.csv')

100%|██████████| 199/199 [14:27<00:00,  4.36s/it]


### Perform Analysis
Run analysis methods as defined in previous analysis code from notebook 4. As written, that analysis code currently takes ~5 minutes per ortholog which could take awhile for the 200 ortholog set. Set a batch size variable to five and do this in chunks of 5 at a time, saving progress inbetween. These files can be loaded and concatenated at a future notebook to provide quantitative analysis.

In [ ]:
BATCH_SIZE = 5

files = sorted([file for file in os.listdir('../input/orthologs/') if file.endswith('.csv')])

for i in tqdm(range(0, len(files), BATCH_SIZE)):
    batch_files = files[i:i + BATCH_SIZE]
    
    batch_df = pd.concat(
        [pd.read_csv(f'../input/orthologs/{file}') for file in batch_files],
        ignore_index=True
    )
    
    # TODO: Recommend revisiting this analysis fn and see if you can speed it up at all
    analysis, counts = match_alias_to_ortholog(subset, batch_df)
    
    batch_number = i // BATCH_SIZE
    analysis.to_csv(f'../output/orthologs/analysis_{batch_number}.csv')

    counts_serialized = {k: int(v) for k, v in counts.items()} # Required to serialize numpy to int, otherwise cannot save to json
    with open(f'../output/orthologs/counts_{batch_number}.json', 'w') as f:
        json.dump(counts_serialized, f, indent=2)
